In [1]:
# Remove unused conflicting packages
!pip uninstall -qqy kfp jupyterlab libpysal thinc spacy fastai ydata-profiling google-cloud-bigquery google-generativeai async-timeout
# Install langgraph and the packages used in this lab.
!pip install -qU 'langgraph==0.3.21' 'langchain-google-genai==2.1.2' 'langgraph-prebuilt==0.1.7' langchain-community git+https://github.com/VergaJU/PMC_download.git
!pip install mysql-connector-python

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 12.1 MB

In [2]:
from google import genai
#from google.genai import types

genai.__version__


'0.2.2'

In [3]:
import os

# List the files in your dataset directory
dataset_path = '/kaggle/input/mirkat-tables-descriptions'
files = os.listdir(dataset_path)
print(files)

['mirkat_tables_columns_descriptions.csv']


In [4]:
import os

# List the files in your dataset directory
dataset_path = '/kaggle/input/mirkat-tables-description-high-level'
files = os.listdir(dataset_path)
print(files)

['mirkat_tables_descriptions.csv']


In [5]:
import re

In [6]:
import os
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [7]:
from google.api_core import retry
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})
if not hasattr(genai.models.Models.generate_content, '__wrapped__'):
  genai.models.Models.generate_content = retry.Retry(
      predicate=is_retriable)(genai.models.Models.generate_content)

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
from typing import TypedDict, List, Literal, Optional, Dict, Any
import json
from langchain_core.messages import (
    BaseMessage,  # Base class for all messages in state['messages']
    HumanMessage, # For messages from the user
    AIMessage,    # For messages from the LLMs
    SystemMessage # For setting system prompts for LLMs
    # ToolMessage will be implicitly handled by LangGraph/ToolNode
)
from langchain_core.tools import tool # Decorator for creating LangChain tools
# --- LangChain LLM Integration ---
# Using Google Generative AI (Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI

# --- LangGraph Components ---
from langgraph.graph import StateGraph, END # Core graph builder and end state marker
from langgraph.prebuilt import ToolNode      # Node specifically for executing tools
from langchain_core.messages import SystemMessage
#import google.ai.generativelanguage as gapic # Use an alias for the Google SDK types

# --- Needed for Tool Implementations ---
import os
from IPython.display import display, Image, Markdown
from pprint import pprint
import pandas as pd
import numpy as np
from PMC_download.retrieve_articles import PapersDownloader as pmc
from Bio import Entrez
import mysql.connector
from mysql.connector import errorcode


from langchain_core.messages import SystemMessage
# Attempt to import directly from the google.generativeai library's types
# If this fails, the underlying import structure might be different, but the principle remains
#try:
#    import google.generativeai.types as genai_types
#except ImportError:
#    # Fallback if the types structure is different in 0.2.2, might need google.ai.generativelanguage
#    # but the key is the structure below
#    print("Could not import google.generativeai.types, trying google.ai.generativelanguage")
#    import google.ai.generativelanguage as genai_types


In [10]:
llm_master = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
sql_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash", # Or gemini-pro, or the model you used for sql_chat
    # Ensure API key is configured via environment variable or otherwise
    temperature=0
    )


In [11]:
llm_master.invoke("hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run-910327ed-5fb7-4466-8003-067a7b7b2792-0', usage_metadata={'input_tokens': 1, 'output_tokens': 11, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

In [12]:
class GraphState(TypedDict):
    messages: List[BaseMessage]
    table: Optional[Dict[str, Any]]
    answer: str
    chunks: list
    supports: list
    finished: bool
    


In [13]:
# # mirkatDB node
    
MIRKAT_USER = UserSecretsClient().get_secret("MIRKAT_USER")
MIRKAT_PSWD = UserSecretsClient().get_secret("MIRNA_PSWD")
config = {
    'user': MIRKAT_USER,
    'password': MIRKAT_PSWD,
    'host': '185.175.171.210',
    'database': 'mirkat',
    'raise_on_warnings': True
}
cnx = None


def connect_sql():
    try:
        cnx = mysql.connector.connect(**config)
        return cnx
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
        else:
            print(err)
        return None
# @tool        
# def list_tables() -> list[str]:
#     """Retrieve the names of all tables in the database."""
#     print(' - DB CALL: list_tables()')
    
#     cursor = db_conn.cursor()
    
#     # Fetch the table names.
#     cursor.execute("SHOW TABLES;")
    
#     tables = cursor.fetchall()
#     cursor.close()
#     return [t[0] for t in tables]

# @tool
# def get_table_schema(table_name: str) -> list[tuple[str, str]]:
#     """Look up the table schema.

#     Returns:
#       List of columns, where each entry is a tuple of (column, type).
#     """
#     print(f' - DB CALL: describe_table({table_name})')

#     cursor = db_conn.cursor()

#     cursor.execute(f"DESCRIBE `{table_name}`;")
    
#     schema = cursor.fetchall()
#     cursor.close()
#     # MySQL returns (Field, Type, Null, Key, Default, Extra), so we extract the first two columns.
#     return [(col[0], col[1]) for col in schema]
# @tool
# def describe_columns(table_name:str) -> list[tuple[str,str]]:
#     """ Looks for the columns in the table table_name and gets the 
#         biological description of the table
#         Args:
#             table_name (str): Name of the table to describe
#         Returns:
#             list[tuple[str,str]]: List of tuples containing column names and their descriptions
#     """
#     # Check if the table name exists in the DataFrame
#     if table_name not in mirkat_columns_desctiption['Table'].values:
#         print(f"Error: Table '{table_name}' not found.")
#         return [("Info", f"No description found for table {table_name}")]

#     # Filter the DataFrame for the specified table name
#     filtered_df = mirkat_columns_desctiption[mirkat_columns_desctiption['Table'] == table_name]

#     # Extract column names and descriptions
#     columns = list(zip(filtered_df['Column Name'], filtered_df['Description']))
    
#     return columns
# @tool   
# def describe_tables() -> list[tuple[str,str]]:
#     """ Looks for the biological description 
#     and returns the description of all the tables
#     """
#     # Extract table names and descriptions
#     tables = list(zip(mirkat_tables_desctiption['Table'], mirkat_tables_desctiption['Description']))
    
#     return tables
# @tool
# def execute_query(sql: str) -> list[list[str]]:
#     """Execute an SQL statement, returning the results."""
#     print(f' - DB CALL: execute_query({sql})')

#     try:
#         cursor = db_conn.cursor()
#         cursor.execute(sql)
#         results = cursor.fetchall()
#         cursor.close()
#         # Convert results to string lists to avoid potential issues with non-serializable types
#         return [[str(item) for item in row] for row in results]
#     except Exception as e:
#         print(f"Error in execute_query: {e}")
#         db_conn.rollback() # Rollback in case of error during query
#         return [[f"Error executing query: {e}"]]
    
db_conn = connect_sql()
# file_path = '/kaggle/input/mirkat-tables-description-high-level/mirkat_tables_descriptions.csv'
# try:
#     mirkat_columns_desctiption = pd.read_csv(file_path)
# except FileNotFoundError:
#     mirkat_columns_desctiption = pd.DataFrame()
#     print(f"Error: File not found at {file_path}")
# file_path = '/kaggle/input/mirkat-tables-description-high-level/mirkat_tables_descriptions.csv' 
# try:
#     mirkat_tables_desctiption = pd.read_csv(file_path)
# except FileNotFoundError:
#     mirkat_tables_desctiption = pd.DataFrame()
#     print(f"Error: File not found at {file_path}")


#db_tools = [list_tables, get_table_schema, describe_columns, describe_tables, execute_query]

# SQL_SYSTEM_INSTRUCTION_CONTENT = """You interact with an MySQL database
# of microRNAs and its targets called mirkat. You will take the users questions and turn them into SQL
# queries using the tools available. Once you have the information you need, you will
# return a Json file. 

# Use list_tables to see what tables are present, get_table_schema to understand the
# schema, describe_tables if you need to know what a table represents, describe_columns if you need to know biological meaning of the columns, and execute_query to issue an SQL SELECT query. 

# Some columns in the tables is not logical, check get_table_schema and describe_columns before set a query if there is not clear the name of the column to query.

# Avoid select all since the tables are huge. 

# mirna, mir, microRNA are the same thing.
# The mature name of the microRNA (specie-mir-n-5p/3p) is the one used in most tables, but people will refer to their short name. For example mir1 instead of hsa-mir-1-5p.
# """
# SQL_SYSTEM_INSTRUCTION = SystemMessage(content=SQL_SYSTEM_INSTRUCTION_CONTENT)

# sql_llm = ChatGoogleGenerativeAI(
#     model="gemini-2.0-flash", # Or gemini-pro, or the model you used for sql_chat
#     # Ensure API key is configured via environment variable or otherwise
#     temperature=0
#     )

# # Bind the DB tools and the specific system instruction
# sql_llm_with_db_tools = sql_llm.bind_tools(db_tools) # WHERE DO I SET THE INSTRUCCTIONS???


In [14]:
file_path = '/kaggle/input/mirkat-tables-descriptions/mirkat_tables_columns_descriptions.csv' 

try:
    mirkat_columns_desctiption = pd.read_csv(file_path)
except FileNotFoundError:
    mirkat_columns_desctiption = pd.DataFrame()
    print(f"Error: File not found at {file_path}")
file_path = '/kaggle/input/mirkat-tables-description-high-level/mirkat_tables_descriptions.csv'
try:
    mirkat_tables_desctiption = pd.read_csv(file_path)
except FileNotFoundError:
    mirkat_tables_desctiption = pd.DataFrame()
    print(f"Error: File not found at {file_path}")

In [15]:
from google.genai import types

def list_tables() -> list[str]:
    """Retrieve the names of all tables in the database."""
    # Include print logging statements so you can see when functions are being called.
    print(' - DB CALL: list_tables()')

    cursor = db_conn.cursor()

    # Fetch the table names.
    cursor.execute("SHOW TABLES;")

    tables = cursor.fetchall()
    return [t[0] for t in tables]


def get_table_schema(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    print(f' - DB CALL: describe_table({table_name})')

    cursor = db_conn.cursor()

    cursor.execute(f"DESCRIBE `{table_name}`;")
    
    schema = cursor.fetchall()
    # MySQL returns (Field, Type, Null, Key, Default, Extra), so we extract the first two columns.
    return [(col[0], col[1]) for col in schema]

def describe_columns(table_name:str) -> list[tuple[str,str]]:
    """ Looks for the columns in the table table_name and gets the 
        biological description of the table
        Args:
            table_name (str): Name of the table to describe
        Returns:
            list[tuple[str,str]]: List of tuples containing column names and their descriptions
    """
    # Check if the table name exists in the DataFrame
    if table_name not in mirkat_columns_desctiption['Table'].values:
        print(f"Error: Table '{table_name}' not found.")
        return []

    # Filter the DataFrame for the specified table name
    filtered_df = mirkat_columns_desctiption[mirkat_columns_desctiption['Table'] == table_name]

    # Extract column names and descriptions
    columns = list(zip(filtered_df['Column Name'], filtered_df['Description']))
    
    return columns
    
def describe_tabes() -> list[tuple[str,str]]:
    """ Looks for the biological description 
    and returns the description of all the tables
    """
    # Extract table names and descriptions
    tables = list(zip(mirkat_tables_desctiption['Table'], mirkat_tables_desctiption['Description']))
    
    return tables

def execute_query(sql: str) -> list[list[str]]:
    """Execute an SQL statement, returning the results."""
    print(f' - DB CALL: execute_query({sql})')

    cursor = db_conn.cursor()

    cursor.execute(sql)
    return cursor.fetchall()

In [16]:
execute_query("Show tables;")

 - DB CALL: execute_query(Show tables;)


[('confidence',),
 ('confidence_score',),
 ('dead_mirna',),
 ('gene_mirna',),
 ('gene_names',),
 ('literature_references',),
 ('mature_database_links',),
 ('mature_database_url',),
 ('mirna',),
 ('mirna_chromosome_build',),
 ('mirna_context',),
 ('mirna_database_links',),
 ('mirna_database_url',),
 ('mirna_literature_references',),
 ('mirna_mature',),
 ('mirna_pre_mature',),
 ('mirna_prefam',),
 ('mirna_seeds',),
 ('mirna_species',),
 ('mirna_tissues',)]

In [17]:
execute_query("describe mirna_database_links;")

 - DB CALL: execute_query(describe mirna_database_links;)


[('auto_mirna', 'int unsigned', 'NO', 'MUL', '0', ''),
 ('auto_db', 'int', 'YES', '', None, ''),
 ('link', 'tinytext', 'NO', '', None, ''),
 ('display_name', 'tinytext', 'NO', '', None, '')]

In [18]:
execute_query("Select * from mirna_seeds limit 3;")

 - DB CALL: execute_query(Select * from mirna_seeds limit 3;)


[('cfa-let-7d', 'UAUACGA', '9615'),
 ('xtr-let-7a', 'GAGGUAG', '8364'),
 ('xtr-let-7b', 'GAGGUAG', '8364')]

In [19]:
# These are the Python functions defined above.
db_tools = [list_tables, get_table_schema, describe_columns, describe_tabes, execute_query]
all_tools = []

# 1. Add your custom database tools
if db_tools: # Check if db_tools is defined and not empty
    all_tools.extend(db_tools) # Use extend since db_tools should be a list

# 2. Add the Google Search tool
google_search_tool = types.Tool(google_search=types.GoogleSearch())
all_tools.append(google_search_tool)


instruction = """You interact with an MySQL database
of microRNAs and its targets called mirkat. You will take the users questions and turn them into SQL
queries. Once you have the information you need, you will
return a Json object. 

If you need additional information use list_tables to see what tables are present, get_table_schema to understand the
schema, describe_tabes is you need to know what a table represents, describe_columns if you need to know biological meaning of the columns, and execute_query to issue an SQL SELECT query.

Avoid select all since the tables are huge. 

Examples:

human query: how many mirs are there?
sql query: SELECT count(*) FROM mirna

human query: Which is the most common seed?
sql query: SELECT seed, count(*) AS count FROM mirna_seeds GROUP BY seed ORDER BY count DESC LIMIT 1

human query: How many mirnas have seed GAGGUAG?
sql query: SELECT count(*) FROM mirna_seeds WHERE seed = 'GAGGUAG'

human query: How many human microRNAs have the seed GAGGUAG
sql query: SELECT COUNT(DISTINCT mm.mature_name) FROM mirna_seeds ms JOIN mirna_mature mm ON ms.auto_mature = mm.mature_name JOIN mirna_pre_mature mpm ON mm.auto_mature = mpm.auto_mature JOIN mirna m ON mpm.auto_mirna = m.auto_mirna JOIN mirna_species sp ON m.auto_species = sp.auto_id WHERE ms.seed = 'GAGGUAG' AND sp.name = 'Homo sapiens'

human query: What are the differences in targets of human mir 106a and mir-106b separed by source?
sql query:  SELECT gm.mrna, gm.mirna_mature, gm.source FROM gene_mirna gm WHERE gm.mirna_mature IN ('hsa-miR-106a-5p', 'hsa-miR-106b-5p') 




"""

client = genai.Client(api_key=GOOGLE_API_KEY)



# Start a chat with automatic function calling enabled.
chat = client.chats.create(
    model="gemini-1.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=db_tools,
    ),
)



In [20]:
response = chat.send_message("hello")

In [21]:
type(response)

google.genai.types.GenerateContentResponse

In [22]:
#mirkat_columns_desctiption[mirkat_columns_desctiption['Table']=='mirna_seeds']
mirkat_tables_desctiption#[mirkat_tables_desctiption["Table"]]
#mirkat_columns_desctiption

,Table,Main Source,Primary Key,Key,Description
0,confidence_score,mirBase,-,-,evaluates the reliability of microRNA annotati...
1,dead_mirna,mirBase,-,-,annotation indicating microRNA entries that ar...
2,literature_references,mirBase,-,-,pertain to the primary research articles and a...
3,mature_database_links,mirBase,-,-,cross-references connecting mature microRNAs ...
4,mature_database_url,mirBase,-,-,refers to the web-based access point for retri...
5,mirna,mirBase,auto_mirna,-,The mirna in their initial state. (the full se...
6,mirna_2_prefam,mirBase,"auto_mirna, auto_prefam",-,maps individual microRNAs to their precursor f...
7,mirna_chromosome_build,mirBase,-,auto_mirna,refers to the genome assembly version used to ...
8,mirna_context,mirBase,-,auto_mirna,refers to the genomic and functional features ...
9,mirna_database_links,mirBase,-,"auto_mirna, link",refer to cross-references connecting microRNA ...


In [23]:
from IPython.display import Markdown

Markdown(chat.send_message("Which microRNA has the highest number of targets? you can create a SQL query that counts the seed per microRNA so you can do it").text)

 - DB CALL: execute_query(SELECT mirna_mature, COUNT(*) AS target_count FROM gene_mirna GROUP BY mirna_mature ORDER BY target_count DESC LIMIT 1)


The microRNA with the highest number of targets is gga-miR-6701-3p, with 9309 targets.


In [24]:
# sql_llm_with_db_tools.invoke("Number of microRNAs in table matrue_mirnas?")

In [25]:
# literature search node
import google.ai.generativelanguage as genai_types
from google.genai import types


# Define the native Google Search tool for the Google SDK
native_google_search_config = [
    types.Tool(
        # Just include the GoogleSearchRetrieval object without parameters
        #google_search_retrieval=genai_types.GoogleSearchRetrieval()
    #)
        google_search_retrieval=types.GoogleSearchRetrieval(
                dynamic_retrieval_config=types.DynamicRetrievalConfig(
                    dynamic_threshold=0.6)) # Or True if you don't want citations
    )
]


llm_literature = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash", # Or gemini-pro, etc. Choose model suitable for reasoning over text
    # Pass the native SDK configuration via model_kwargs
    temperature=0.0,
    model_kwargs={
        "tools": native_google_search_config,
        "tool_config": {"function_calling_config": "AUTO"} # Let model decide when to search
    }
)


# Define the System Prompt/Instruction for this LLM's role
# This guides the LLM on how to behave and use the tool
LITERATURE_SYSTEM_INSTRUCTION_CONTENT = """You are a research assistant. Your goal is to answer user questions accurately, leveraging your built-in web search capabilities to find factual, up-to-date information when necessary for the query.

1.  Receive the user's question from the conversation history.
2.  Formulate a concise, relevant query for the  based on the core information need in the user's question. Use only peer review documentation such as published articles or books.
3.  **use your integrated web search functionality.**
4.  Wait for the tool to return results (they will appear as a ToolMessage in the history).
5.  **Analyze the search results provided in the ToolMessage.**
6.  Synthesize a final answer to the user's original question, strictly using the information from the search results.
7.  If the search results are empty, indicate that no information was found in the knowledge base.
8.  If the tool returned an error, report that.
9.  **DO NOT use your general knowledge.** Your answers MUST be grounded in the provided search results. Start your final answer (after receiving tool results) with "Based on the search results:"
"""
LITERATURE_SYSTEM_INSTRUCTION = SystemMessage(content=LITERATURE_SYSTEM_INSTRUCTION_CONTENT)

# Bind the tool and system message to this specific LLM
# When invoked, this LLM will automatically have the system message and know about the tool.

In [26]:
Markdown(llm_literature.invoke("Which microRNA has to do with sarcopenia??").content)

There isn't one single microRNA (miRNA) definitively linked to sarcopenia.  Sarcopenia is a complex process involving multiple pathways and genetic factors.  Research has implicated several miRNAs in its development and progression, but their roles are often context-dependent and not fully understood.

Some miRNAs frequently studied in the context of sarcopenia include:

* **miR-29:**  Studies suggest it's involved in muscle atrophy and fibrosis.
* **miR-1:**  Implicated in muscle regeneration and differentiation.  Dysregulation may contribute to sarcopenia.
* **miR-23a/27a/24-2:**  These miRNAs are often studied together as they are involved in muscle growth and differentiation.  Changes in their expression are associated with age-related muscle loss.
* **miR-21:**  This miRNA is involved in various cellular processes, and its role in sarcopenia is complex and not fully elucidated.  Some studies show increased expression in sarcopenic muscle.
* **miR-34a:**  Associated with muscle aging and atrophy.

It's crucial to understand that the involvement of these miRNAs is often intertwined with other factors, such as inflammation, oxidative stress, and hormonal changes.  The specific miRNAs and their effects can vary depending on the individual, the stage of sarcopenia, and other contributing factors.  Further research is needed to fully clarify the roles of individual miRNAs in sarcopenia.

In [27]:
response=llm_literature.invoke("who is Bombardiro cocodiro?")
response

AIMessage(content='There is no known public figure or character named "Bombardiro Cocodiro."  It\'s possible this is a misspelling, a fictional character from a niche work, or a completely made-up name.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run-a1319ffa-d562-456f-8ae8-ed3e37025d12-0', usage_metadata={'input_tokens': 10, 'output_tokens': 46, 'total_tokens': 56, 'input_token_details': {'cache_read': 0}})

In [28]:
type(response)

langchain_core.messages.ai.AIMessage

In [29]:
# master node

ORIGINAL_MIRNA_SYSINT_CONTENT = """
You are MiRNA Researcher Assistant, basically another high level researcher. You can help with information regarding the microRNAs and their context. You have access to the miRKatDB with includes the infromation from miRBase, targetScan, mirnaTissueAtlas and other relevant microRNA databases. 
Aditionally, you can search the web to increase the context of the microRNAs, their functions, mechanisms of actions or any related to the biology. 
If the conversation is getting off topic, you must inform the user. If there is no more microRNA releted queries, finish the conversation.
""" # Replace with your actual original content

ROUTING_INSTRUCTIONS = """
## Routing Instructions:
Based on the user's latest message, analyze the request and decide the *next immediate step*. Respond ONLY with ONE of the following keywords, nothing else:

1.  `***ROUTE_TO_SQL***`: If the question *clearly* requires specific data retrieval from the miRKat database (e.g., list targets, find miRNA by seed, check expression levels, database schema questions).
2.  `***ROUTE_TO_LITERATURE***`: If the question asks for functional information, mechanisms, biological context, recent research, definitions, or information likely found in scientific papers or reviews that is not simple structured data.
3.  `***ANSWER_DIRECTLY***`: If you can answer the question directly based on the conversation history or general knowledge appropriate for this assistant, OR if you need to ask the user a clarifying question before proceeding.
4.  `***FINISH***`: If the user indicates they want to end the conversation (e.g., "thanks, that's all", "goodbye").

**Example Decisions:**
- User: "What are the validated targets of hsa-let-7a?" -> `***ROUTE_TO_SQL***`
- User: "Tell me about the role of miR-21 in cancer." -> `***ROUTE_TO_LITERATURE***`
- User: "Thanks for the target list. Can you explain target prediction algorithms?" -> `***ROUTE_TO_LITERATURE***`
- User: "Which database tables store tissue expression?" -> `***ROUTE_TO_SQL***`
- User: "What is a microRNA?" -> `***ANSWER_DIRECTLY***` (or `***ROUTE_TO_LITERATURE***` if wanting detailed explanation)
- User: "Okay, thank you! Bye" -> `***FINISH***`
- User: "Can you search for papers on lncRNAs?" -> `***ROUTE_TO_LITERATURE***` (It's related biology)
- User: "What's the weather?" -> `***ANSWER_DIRECTLY***` (Acknowledge off-topic, maybe offer to return to miRNAs)
"""

# Combine them (adjust formatting as needed for your LLM)
MIRNA_ASSISTANT_SYSINT_CONTENT = ORIGINAL_MIRNA_SYSINT_CONTENT + ROUTING_INSTRUCTIONS
ORIGINAL_MIRNA_SYSINT_CONTENT_MESSAGE = SystemMessage(ORIGINAL_MIRNA_SYSINT_CONTENT)
MIRNA_ASSISTANT_SYSTEM_MESSAGE = SystemMessage(content=MIRNA_ASSISTANT_SYSINT_CONTENT) # Create SystemMessage object

WELCOME_MSG = "Hello there. Please ask me your microRNA related questions. I have access to miRKat database and general web search."



In [30]:
llm_master.invoke("hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run-2d12fb99-23da-4ab6-95f6-660d0272153d-0', usage_metadata={'input_tokens': 1, 'output_tokens': 11, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

In [31]:
# nodes
from google.genai.types import GenerateContentResponse

def human_node(state: GraphState) -> GraphState:
    """Display the last model message to the user, and rec
    eive the user's input."""
    print("\n--- ENTERING: human_node ---")
    last_msg = state["messages"][-1]
    answer = state["answer"]
    support = None
    chunks = None
    print(F"----- ANSWER: {answer} -------")
    if isinstance(last_msg, AIMessage) or isinstance(last_msg, GenerateContentResponse):
        if answer:
            print("Assistant:", answer)
            state["answer"] = None
            print()
        else:
            print("Assistant:", last_msg.content)
    print("="*30)

    user_input = input("User: ")

    # If it looks like the user is trying to quit, flag the conversation
    # as over.
    if user_input.strip().lower() in {"q", "quit", "exit", "goodbye", "bye", "thanks that's all"}:
        state["finished"] = True

    #return state | {"messages": [("user", user_input)]}
    return {
        "messages": state["messages"] +[HumanMessage(content=user_input)],
        "table": state["table"],
        "answer": state["answer"],
        "finished": state["finished"]}

def chatbot_with_tools(state: GraphState) -> GraphState:
    """The chatbot with tools. A simple wrapper around the model's own chat interface."""
    print("\n--- ENTERING: master_node ---")

    
    messages = state['messages']

    # Check if this is the very first turn (no messages yet)
    if not messages:
        # Generate the welcome message directly
        print("--- Generating Welcome Message ---")
        response = AIMessage(content=WELCOME_MSG)
    else:
        # Normal operation: Invoke the master LLM for routing/response
        print("--- Calling Master Router LLM ---")
        # Always invoke with the system message + current history
        #print(f"--- Message going to the llm_master: {messages}---")
        response = llm_master.invoke([MIRNA_ASSISTANT_SYSTEM_MESSAGE] + messages)
        if response.content.strip() == "***ANSWER_DIRECTLY***":
            response = llm_master.invoke([ORIGINAL_MIRNA_SYSINT_CONTENT_MESSAGE] + messages)
            answer = response.content
        print(f"--- Master Router Raw Response: {response.content} ---")

    # Update state
    return {
        **state, # Preserve other state fields
        "messages": state["messages"] + [response] # Add the router's decision/response
    }

def sql_processor_node(state: GraphState) -> GraphState:
    """The sql llm that will check for the sql questions and get a json file in response."""
    print("--- Calling SQL Processor Node ---")
    messages = [state['messages'][-2].content]
    if not messages:
        # Should ideally not happen if routing is correct
        print("Warning: SQL processor called with no messages.")
        # Return unchanged state or add an error message? For now, return unchanged.
        return state
    #print("The message sent to the SQL node is: ", messages)
    response = chat.send_message(messages)

    #response = sql_llm_with_db_tools.invoke([SQL_SYSTEM_INSTRUCTION] + messages)
    #print(f"--- SQL Processor LLM Response: {response} ---")
    
    
    new_answer = state.get("answer", "")
    
    if isinstance(response, AIMessage) and response.content and not response.tool_calls:
         new_answer = response.content # Update answer if it's a direct text response
    elif isinstance(response, GenerateContentResponse):
        new_answer = response.candidates[0].content.parts[0].text
    elif isinstance(response, str):
        new_answer = response
    new_messages = messages + [AIMessage(content=new_answer)]
    print(f"--- Answer from SQL Processor LLM Response: {new_answer} ---")
    return {
        "messages": new_messages,
        "table": state.get("table", []), # Use .get for safety
        "answer": new_answer, # Return the potentially updated answer
        "finished": state.get("finished", False), # Use .get for safety
    }
    
def literature_search_node(state: GraphState) -> GraphState:
    """
    Invokes the literature search LLM. This LLM is expected to either:
    1. Request the 'ground_search' tool.
    2. Synthesize an answer if 'ground_search' results are already in the message history.
    """
    print("--- Calling Literature Search Node ---")
    # Get the current message history safely
    messages = state.get('messages', [])

    if not messages:
        print("Warning: Literature Search node called with no messages.")
        # Return state unchanged or add an error message? For now, unchanged.
        return state

    response = llm_literature.invoke([LITERATURE_SYSTEM_INSTRUCTION] + messages)

    print(f"--- Literature LLM Response: {response} ---")
    


    # Append the LLM's response (which could be a tool call or a final answer)
    new_messages = messages + [response]

    # Update the 'answer' field ONLY if this response is the final synthesized text
    # and NOT a request to call a tool.
    new_answer = state.get("answer", "") # Keep previous answer by default
    if isinstance(response, AIMessage) and response.content and not response.tool_calls:
         print("--- Literature Node updating final answer ---")
         new_answer = response.content

    # Return the updated state
    return {
        "messages": new_messages,
        "table": state.get("table", []), # Pass through unchanged state fields
        "answer": new_answer,           # Update answer if synthesized
        "finished": state.get("finished", False),
    }
all_tools = db_tools # Add literature search tools here if they were LangChain tools
tool_node = ToolNode(all_tools)

In [32]:
# Define node names for clarity
HUMAN_NODE = "human_node"
CHATBOT_NODE = "chatbot_router"
SQL_NODE = "sql_processor_node"
LITERATURE_NODE = "literature_search_node"
TOOL_NODE = "execute_tools" # Name for the ToolNode instance


In [33]:
# edges
def maybe_exit_human_node(state: GraphState) -> Literal["chatbot", "__end__"]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"
def route_after_human(state: GraphState) -> Literal["chatbot_router", "__end__"]:
    """
    Determines the next step after the human node.
    If the 'finished' flag is set, ends the graph.
    Otherwise, directs the conversation to the main chatbot.
    """
    print("\n--- ROUTING: route_after_human ---")
    if state.get("finished", False):
        print("--- Routing: Human to END ---")
        return END
    else:
        print("--- Routing: Human to Chatbot ---")
        return CHATBOT_NODE

# Router 2: After the Main Chatbot/Router (`chatbot_with_tools`)
def route_chatbot_decision(state: GraphState) -> Literal["sql_processor_node", "literature_search_node","human_node", "__end__"]:
    """
    Inspects the last message from the main chatbot (`chatbot_with_tools`)
    and decides where to route the conversation next.
    """
    print("\n--- ROUTING: route_chatbot_decision ---")
    messages: List[BaseMessage] = state['messages']
    if not messages:
        # Should not happen in a normal flow, but handle defensively
        print("--- Routing Error: No messages found in route_chatbot_decision ---")
        return END # Or raise error

    last_message = messages[-1]
    
    if not isinstance(last_message, AIMessage) :
        # If the last message isn't from the AI, something is wrong in the flow
        print(f"--- Routing Warning: Expected AIMessage, got {type(last_message)}. Routing to Human. ---")
        return HUMAN_NODE
        
    content = last_message.content.strip()
    
    # Check for routing keywords first
    if "***ROUTE_TO_SQL***" in content:
        print("--- Routing: Master Router to SQL Processor ---")
        # Optionally modify the message to be less robotic before sending to SQL node
        # state['messages'][-1].content = "Okay, I need to query the database for that."
        return SQL_NODE
    elif "***ROUTE_TO_LITERATURE***" in content:
        print("--- Routing: Master Router to Literature Searcher ---")
        # state['messages'][-1].content = "Okay, I need to search the literature for that."
        return LITERATURE_NODE
    elif "***FINISH***" in content or state.get("finished"): # Check flag too
        print("--- Routing: Master Router to END ---")
        return END
    elif "***ANSWER_DIRECTLY***" in content:
         print("--- Routing: Master Router to Human (Direct Answer) ---")
         # Remove the keyword itself before showing to human
         state['messages'][-1].content = content.replace("***ANSWER_DIRECTLY***", "").strip()
         # If the content is *only* the keyword, maybe add a placeholder?
         #if not state['messages'][-1].content:
         x = state['messages'][-2].content#.candidates[0].content.parts[0].text
         print (f"--- The messages directly was: {x}")
         answer = llm_master.invoke(x) #"Okay, let me answer that." # Or similar
         state['messages'][-1].content = re.sub(r'[*_`~#\[\]()]', '', answer.content)
         print (f"--- The answer directly was: {answer}")
         state['answer'] = answer#.response.candidates[0].content.parts[0].text
         return HUMAN_NODE
    else:
         # Assume it's a direct answer or clarification question if no keyword is found
         print("--- Routing: Master Router to Human (Direct Answer) ---")
         # Remove potential keywords just in case they were missed but shouldn't be shown
         state['messages'][-1].content = content.replace("***ROUTE_TO_SQL***", "").replace("***ROUTE_TO_LITERATURE***", "").replace("***FINISH***", "").replace("***ANSWER_DIRECTLY***", "").strip()
         return HUMAN_NODE

# Router 3: After a Specialist Processor Node (`sql_processor_node`, `literature_search_node`)
def route_processor_output(state: GraphState) -> Literal["human_node", "__end__"]:
    """
    Inspects the last message from a specialist processor node.
    Routes to 'tools' if a tool call was made (e.g., query_database, ground_search).
    Routes to 'human_node' if a final synthesized answer was provided.
    """
    print("\n--- ROUTING: route_processor_output ---")
    messages: List[BaseMessage] = state['messages']
    if not messages:
        print("--- Routing Error: No messages found in route_processor_output ---")
        return END

    last_message = messages[-1]

    #if not isinstance(last_message, AIMessage):
    #    print(f"--- Routing Warning: Expected AIMessage from processor, got {type(last_message)}. Routing to Human. ---")
    #    return HUMAN_NODE
    # Otherwise, the processor provided its final synthesized answer
    #else:
    #    print("--- Routing: Processor to Human ---")
    return HUMAN_NODE
        
def route_after_tools(state: GraphState) -> Literal["sql_processor_node", "literature_search_node","human_node"]:
    """ Routes back to the specialist node that originally called the tool OR to human if unclear."""
    print("\n--- ROUTING: route_after_tools ---")
    messages = state['messages']
    # The last message is the ToolMessage with results
    # The second to last message *should* be the AIMessage that made the tool call
    if len(messages) < 2:
         print("--- Routing Warning: Tool execution happened without prior AI message? Routing to Human. ---")
         return HUMAN_NODE # Should not happen

    ai_message_that_called_tool = messages[-2]

    # This is heuristic: Check which LLM likely generated the tool call
    # A more robust way might be to add metadata to the state indicating the caller.
    # Simple approach: Check the tools called. If DB tools, assume SQL node. If search, assume Literature.

    # Check if the tool call originated from SQL LLM (by checking tool names)
    db_tool_names = {t.name for t in db_tools}
    called_tool_names = {call['name'] for call in ai_message_that_called_tool.tool_calls}

    if any(name in db_tool_names for name in called_tool_names):
         print("--- Routing: Tools back to SQL Processor ---")
         return SQL_NODE
    # Check if the tool call was Google Search (handled implicitly by LangChain for native tools)
    elif any(call['name'].lower() == 'googlesearchretrieval' for call in ai_message_that_called_tool.tool_calls):
         print("--- Routing: Tools back to Literature Searcher ---")
         return LITERATURE_NODE
    else:
         # Fallback if the origin is unclear
         print(f"--- Routing Warning: Tool caller unclear ({called_tool_names}). Routing to Human. ---")
         # Add a message indicating confusion?
         state['messages'].append(SystemMessage(content="(System: Unclear which process should handle the tool results. Displaying results directly.)"))
         return HUMAN_NODE


In [34]:
# --- Build the Graph ---
workflow = StateGraph(GraphState)

# Add Nodes
workflow.add_node(HUMAN_NODE, human_node)
workflow.add_node(CHATBOT_NODE, chatbot_with_tools)
workflow.add_node(SQL_NODE, sql_processor_node)
workflow.add_node(LITERATURE_NODE, literature_search_node)
workflow.add_node(TOOL_NODE, tool_node)

# --- Define Edges ---

# 1. Entry Point (Where the graph starts)
workflow.set_entry_point(CHATBOT_NODE) # Start with a hello input

# 2. From Human Node
workflow.add_conditional_edges(
    HUMAN_NODE,
    route_after_human, # Function to decide next step
    {
        CHATBOT_NODE: CHATBOT_NODE, # If function returns "chatbot_with_tools", go there
        END: END                   # If function returns "__end__", finish
    }
)

# 3. From Main Chatbot Node
workflow.add_conditional_edges(
    CHATBOT_NODE,
    route_chatbot_decision, # Function to decide based on chatbot output
    {
        SQL_NODE: SQL_NODE,                 # Route to SQL processor
        LITERATURE_NODE: LITERATURE_NODE,   # Route to Literature searcher
        TOOL_NODE: TOOL_NODE,               # Route to execute chatbot's tools (get_menu)
        HUMAN_NODE: HUMAN_NODE,             # Route to show chatbot's direct answer
        END: END                           # Route to end (though usually handled via human)
    }
)

# 4. From SQL Processor Node
workflow.add_conditional_edges(
    SQL_NODE,
    route_processor_output, # Function to decide based on SQL processor output
    {
        TOOL_NODE: TOOL_NODE,   # Route to execute SQL tools (query_database)
        HUMAN_NODE: HUMAN_NODE  # Route to show final SQL answer
    }
)

# 5. From Literature Search Node
workflow.add_conditional_edges(
   LITERATURE_NODE,
   route_processor_output, # Function to decide based on Literature processor output
   {
       TOOL_NODE: TOOL_NODE,   # Route to execute literature tools (ground_search)
       HUMAN_NODE: HUMAN_NODE  # Route to show final literature answer
   }
)

# 6. From Tool Node - Route back to the appropriate processor
workflow.add_conditional_edges(
    TOOL_NODE,
    route_after_tools,
    {
        SQL_NODE: SQL_NODE,
        LITERATURE_NODE: LITERATURE_NODE,
        HUMAN_NODE: HUMAN_NODE # Fallback route
    }
)

In [35]:
# Optional: Add memory/checkpointing
app = workflow.compile()


In [36]:
#Image(app.get_graph().draw_mermaid_png())

In [37]:
# Initial state with a welcome message
initial_state = {
    "messages": [], # <-- Empty list
    "table": None,
    "answer": "",
    "finished": False
}
current_state = initial_state


In [38]:
config = {"recursion_limit": 100}

# Remember that this will loop forever, unless you input `q`, `quit` or one of the
# other exit terms defined in `human_node`.
# Uncomment this line to execute the graph:


In [ ]:
app.invoke(current_state, config)


--- ENTERING: master_node ---
--- Generating Welcome Message ---

--- ROUTING: route_chatbot_decision ---
--- Routing: Master Router to Human (Direct Answer) ---

--- ENTERING: human_node ---
----- ANSWER:  -------
Assistant: Hello there. Please ask me your microRNA related questions. I have access to miRKat database and general web search.


User:  How many different seeds do mirnas that target SIRT1 are?



--- ROUTING: route_after_human ---
--- Routing: Human to Chatbot ---

--- ENTERING: master_node ---
--- Calling Master Router LLM ---
--- Master Router Raw Response: ***ROUTE_TO_SQL*** ---

--- ROUTING: route_chatbot_decision ---
--- Routing: Master Router to SQL Processor ---
--- Calling SQL Processor Node ---
 - DB CALL: execute_query(SELECT COUNT(DISTINCT seed) FROM mirna_seeds ms JOIN mirna_mature mm ON ms.auto_mature = mm.mature_name JOIN gene_mirna gm ON mm.mature_name = gm.mirna_mature WHERE gm.mrna = 'SIRT1')
--- Answer from SQL Processor LLM Response: There are 20 different seeds in microRNAs that target SIRT1.
 ---

--- ROUTING: route_processor_output ---

--- ENTERING: human_node ---
----- ANSWER: There are 20 different seeds in microRNAs that target SIRT1.
 -------
Assistant: There are 20 different seeds in microRNAs that target SIRT1.



In [ ]:
graph = app.get_graph()
mermaid_text = graph.draw_mermaid()

In [ ]:
print(f"Mermaid diagram length: {len(mermaid_text)} characters")


In [ ]:
try:
    img = Image(graph.draw_mermaid_png()) # Default 10s timeout
    display(img)
except Exception as e:
    print(f"--- Error during PNG generation ---")
    print(e)
    print("--- Trying with longer timeout ---")
    try:
        # Try recreating the underlying call with more timeout
        # Note: This is an approximation, the actual internal call might differ
        import base64
        import requests
        graph_bytes = mermaid_text.encode("utf8")
        base64_bytes = base64.b64encode(graph_bytes)
        base64_string = base64_bytes.decode("ascii")
        render_url = f"https://mermaid.ink/img/{base64_string}"
        response = requests.get(render_url, timeout=30) # Longer timeout
        response.raise_for_status() # Check for HTTP errors
        img_png = Image(response.content)
        display(img_png)
        print("--- Successfully rendered with longer timeout ---")
    except Exception as e2:
        print(f"--- Failed even with longer timeout: {e2} ---")

In [ ]:
while not current_state.get("finished", False):
     # The human_node handles printing the last message and getting input
     # It's the first node after the initial welcome message.
     current_state = app.invoke(current_state, config)
     # Add a small check to prevent infinite loops if something goes wrong
     if len(current_state.get("messages", [])) > (config["recursion_limit"] - 5): # Safety break
         print("\n--- Max Recursion Limit Approaching ---")
         print("Final State:", current_state)
         break


print("\n--- Conversation Ended ---")
# Display final state if needed
# pprint(current_state)

In [ ]:
current_state